In [ ]:
# 1. Import libraries
import pandas as pd  # pandas = used to work with tables (dataframes)
import numpy as np  # numpy = used for math operations and arrays
import matplotlib.pyplot as plt  # used for creating graphs (charts)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV  # split data + tuning models
from sklearn.preprocessing import StandardScaler  # used to normalize (scale) data

In [ ]:
# Models
from sklearn.linear_model import LogisticRegression  # Logistic Regression model
from sklearn.tree import DecisionTreeClassifier  # Decision Tree model
from sklearn.neighbors import KNeighborsClassifier  # KNN model

In [ ]:
# Evaluation
from sklearn.metrics import (
    accuracy_score,  # checks how many predictions are correct
    classification_report,  # detailed report (precision, recall, f1)
    confusion_matrix,  # table of correct vs wrong predictions
    ConfusionMatrixDisplay,  # display confusion matrix as chart
    roc_curve,  # used to draw ROC curve
    roc_auc_score  # AUC score (how good model is overall)
)

In [ ]:
# 2. Load dataset
df = pd.read_csv("phishing.csv")  # load CSV file into dataframe

In [ ]:
# 3. Dataset Exploration
print("First 5 rows:")  # print title
print(df.head())  # show first 5 rows of data

In [ ]:
print("\nDataset Info:")  # print info title
print(df.info())  # show column types and missing values

In [ ]:
print("\nStatistics:")  # print statistics title
print(df.describe())  # show mean, min, max, etc.

In [ ]:
# 4. Handle missing values
df = df.fillna(df.median(numeric_only=True))  # fill missing values with median (middle value)

In [ ]:
# 5. Separate features and target
# ⚠️ Change 'Result' if your target column name is different
X = df.drop("class", axis=1)  # features = all columns except "class"
y = df["class"]  # target = "class" column

In [ ]:
# Convert labels (-1 → 0)
y = y.replace(-1, 0)  # replace -1 with 0 (binary classification)

In [ ]:
# 6. Train-test split
# Using 80/20 train-test split and 3-fold cross-validation (GridSearchCV) for tuning
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # 80% training, 20% testing
)

In [ ]:
# 7. Normalization
scaler = StandardScaler()  # create scaler
X_train = scaler.fit_transform(X_train)  # fit and scale training data
X_test = scaler.transform(X_test)  # scale test data using same scaler

=========================
BEFORE TUNING (DEFAULT MODELS)
=========================

In [ ]:
# Logistic Regression BEFORE tuning
model1_default = LogisticRegression(max_iter=1000)  # create model
model1_default.fit(X_train, y_train)  # train model
y_pred1_default = model1_default.predict(X_test)  # make predictions

In [ ]:
print("\nLogistic Regression BEFORE tuning:", accuracy_score(y_test, y_pred1_default))  # print accuracy

In [ ]:
# Decision Tree BEFORE tuning
model2_default = DecisionTreeClassifier(random_state=42)  # create model
model2_default.fit(X_train, y_train)  # train model
y_pred2_default = model2_default.predict(X_test)  # predict

In [ ]:
print("Decision Tree BEFORE tuning:", accuracy_score(y_test, y_pred2_default))  # accuracy

In [ ]:
# KNN BEFORE tuning
model3_default = KNeighborsClassifier()  # create model
model3_default.fit(X_train, y_train)  # train
y_pred3_default = model3_default.predict(X_test)  # predict

In [ ]:
print("KNN BEFORE tuning:", accuracy_score(y_test, y_pred3_default))  # accuracy

In [ ]:
# =========================
# MODEL 1: Logistic Regression (with tuning)
# =========================
param_grid_lr = {
    "C": [0.001, 0.01, 0.1, 1, 10],  # regularization strength values
    "solver": ["liblinear", "lbfgs"]  # algorithm to train model
}

In [ ]:
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000),  # model
    param_grid_lr,  # parameters to try
    cv=3,  # 3-fold cross-validation
    scoring="accuracy"  # measure performance
)

In [ ]:
grid_lr.fit(X_train, y_train)  # train and find best parameters

In [ ]:
model1 = grid_lr.best_estimator_  # best model after tuning

In [ ]:
print("\nBest Logistic Regression Parameters:", grid_lr.best_params_)  # print best parameters

In [ ]:
y_pred1 = model1.predict(X_test)  # predict using tuned model

In [ ]:
# =========================
# MODEL 2: Decision Tree (with tuning)
# =========================
param_grid = {
    "max_depth": [5, 10, 15, 20],  # tree depth
    "min_samples_split": [2, 5, 10],  # min samples to split
    "min_samples_leaf": [1, 2, 5],  # min samples in leaf
    "criterion": ["gini", "entropy"]  # splitting method
}

In [ ]:
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),  # model
    param_grid,  # parameters
    cv=3,
    scoring="accuracy"
)

In [ ]:
grid.fit(X_train, y_train)  # train

In [ ]:
model2 = grid.best_estimator_  # best model

In [ ]:
print("\nBest Decision Tree Parameters:", grid.best_params_)  # print best parameters

In [ ]:
y_pred2 = model2.predict(X_test)  # predict

In [ ]:
# =========================
# MODEL 3: KNN (with tuning)
# =========================
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],  # number of neighbors
    "weights": ["uniform", "distance"]  # weight type
}

In [ ]:
grid_knn = GridSearchCV(
    KNeighborsClassifier(),  # model
    param_grid_knn,
    cv=3,
    scoring="accuracy"
)

In [ ]:
grid_knn.fit(X_train, y_train)  # train

In [ ]:
model3 = grid_knn.best_estimator_  # best model

In [ ]:
print("\nBest KNN Parameters:", grid_knn.best_params_)  # print best parameters

In [ ]:
y_pred3 = model3.predict(X_test)  # predict

================================
BEFORE vs AFTER (TABLE + CHART)
================================

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score  # import metrics

In [ ]:
# Function to calculate all metrics
def get_metrics(y_true, y_pred, model, X):
    return [
        accuracy_score(y_true, y_pred),  # accuracy
        precision_score(y_true, y_pred),  # precision
        recall_score(y_true, y_pred),  # recall
        f1_score(y_true, y_pred),  # f1-score
        roc_auc_score(y_true, model.predict_proba(X)[:, 1])  # auc score
    ]

In [ ]:
# Store results
rows = []  # empty list

In [ ]:
def add_row(model_name, stage, values):
    rows.append([
        model_name,  # model name
        stage,  # before or after
        round(values[0], 4),  # accuracy
        round(values[1], 4),  # precision
        round(values[2], 4),  # recall
        round(values[3], 4),  # f1
        round(values[4], 4)  # auc
    ])

(Rest of code continues same logic — collecting results, printing table, plotting graphs, confusion matrix, ROC curve)